# Demo 1 - Pre-processing + Linguistic Analysis
## Introduction to NLP - Part 1

---

**What we will do in this demo:**

We take a set of short customer reviews - plain text - and run them through a full NLP
pipeline, one step at a time.

By the end you will have seen exactly what happens inside every NLP system when it turns
raw text into something a machine can work with:

- Sentence segmentation, tokenization, and regex cleaning
- Part-of-speech tagging, dependency parsing, and chunking
- Stopword removal and lemmatization - both using **NLTK**
- Named entity recognition
- One single, reusable preprocessing function that ties everything together


---
## Step 0 - Install and Load spaCy and NLTK

We need spaCy for the linguistic analysis steps (POS tagging, dependency parsing,
chunking, named entity recognition) and NLTK for stopword removal and lemmatization.

spaCy's English model is a pre-trained file that already knows English grammar, common
words, and how to tag parts of speech. NLTK gives us curated word lists (stopwords) and a
dictionary-based lemmatizer (WordNet).


In [1]:
# Install spaCy and NLTK - run this cell once
!pip install spacy nltk --quiet
!python -m spacy download en_core_web_sm --quiet

print("Done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 82.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Done.


In [2]:
import re
import spacy
import nltk
from collections import Counter

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# NLTK data needed for tokenizing, POS tagging, stopwords, and lemmatization
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

# Load the English model into a variable called nlp
# When we call nlp(some_text) it runs the full spaCy pipeline automatically
nlp = spacy.load("en_core_web_sm")

lemmatizer = WordNetLemmatizer()

print("spaCy version:", spacy.__version__)
print("NLTK version:", nltk.__version__)
print("spaCy model loaded successfully.")
print("Pipeline steps active:", nlp.pipe_names)

spaCy version: 3.8.16
NLTK version: 3.9.1
spaCy model loaded successfully.
Pipeline steps active: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


---
## Step 1 - Our Review Dataset

Instead of pulling reviews from an external corpus, we define our own small set of
customer reviews. This is the same running example used throughout the Embeddings and
Transformer sessions, so the whole course stays consistent.

Notice the mix:

- Different domains - electronics, restaurant, movie, delivery
- Different sentiment - positive, negative, and mixed
- Different word forms - `excellent`, `stunning`, `arrived`, `delivers`


In [3]:
reviews = [
    "The camera quality is excellent and photos look stunning in low light.",
    "Battery life is terrible, it barely lasts half a day.",
    "The waiter was friendly but the pasta arrived cold and bland.",
    "An absolutely brilliant performance kept me hooked until the end.",
    "Delivery was fast, packaging was neat, and the product works great.",
]

print(f"We have {len(reviews)} reviews in our dataset:")
print()
for i, r in enumerate(reviews, start=1):
    print(f"  R{i}: {r}")

We have 5 reviews in our dataset:

  R1: The camera quality is excellent and photos look stunning in low light.
  R2: Battery life is terrible, it barely lasts half a day.
  R3: The waiter was friendly but the pasta arrived cold and bland.
  R4: An absolutely brilliant performance kept me hooked until the end.
  R5: Delivery was fast, packaging was neat, and the product works great.


---
## Step 2 - NLTK Stopwords

**What is a stopword?**

Stopwords are extremely common words - `"the"`, `"is"`, `"and"`, `"a"` - that carry very
little meaning on their own. Most NLP pipelines remove them before the "real" analysis,
since they add noise without adding signal.

NLTK ships a curated English stopword list. Let's load it and look at it directly.


In [4]:
nltk_stopwords = stopwords.words("english")

print(f"NLTK's English stopword list has {len(nltk_stopwords)} words.")
print()
print("First 30 stopwords:")
print(nltk_stopwords[:30])
print()
print("Full list:")
print(sorted(nltk_stopwords))

NLTK's English stopword list has 198 words.

First 30 stopwords:
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't"]

Full list:
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", "he's", 'her', 'here', 'hers', 'herself', 'him', 'himself', 'his', 'how', 'i', "i'd", "i'll", "i'm", "i've", 'if', 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "i

---
## Step 3 - Sentence Segmentation

**What is it?**

Sentence segmentation means splitting a block of text into individual sentences.

This sounds simple but it is trickier than it looks. Consider:

- `"Dr. Smith works at St. Mary's hospital."` - the full stop after `Dr` is NOT a sentence
  boundary.

spaCy handles these edge cases automatically as part of its pipeline.


In [5]:
# We will work with Review 3 for the next several steps - it has a good mix of
# adjectives, a conjunction, and a clear structure.
# raw_review = reviews[2]
raw_review = "Battery life is terrible, it barely lasts half a day. The camera quality is excellent and photos look stunning in low light."
doc = nlp(raw_review)
sentences = list(doc.sents)

print(f"spaCy found {len(sentences)} sentence(s) in Review 3:")
print()
for i, sentence in enumerate(sentences, start=1):
    print(f"  Sentence {i}: {sentence.text.strip()}")

spaCy found 2 sentence(s) in Review 3:

  Sentence 1: Battery life is terrible, it barely lasts half a day.
  Sentence 2: The camera quality is excellent and photos look stunning in low light.


In [6]:
# We will use the first (and only) sentence for the remaining single-sentence steps
working_sentence = sentences[0]

print("Working sentence for the rest of Demo 1:")
print()
print(" ", working_sentence.text.strip())

Working sentence for the rest of Demo 1:

  Battery life is terrible, it barely lasts half a day.


---
## Step 4 - Tokenization

**What is it?**

Tokenization splits text into individual units called **tokens**. In most cases a token
is a word, but punctuation marks are also separate tokens.

**Example:**

```
"I loved it!"  -->  ["I",  "loved",  "it",  "!"]
```

Machines cannot work with raw characters directly - tokens are the first unit they can
count, compare, and process.


In [7]:
# spaCy tokenizes automatically when you call nlp()
tokens = [token.text for token in working_sentence]

print("Tokens in the working sentence (spaCy):")
print()
for i, t in enumerate(tokens, start=1):
    print(f"  Token {i:>2}: {repr(t)}")

print()
print(f"Total tokens: {len(tokens)}")

Tokens in the working sentence (spaCy):

  Token  1: 'Battery'
  Token  2: 'life'
  Token  3: 'is'
  Token  4: 'terrible'
  Token  5: ','
  Token  6: 'it'
  Token  7: 'barely'
  Token  8: 'lasts'
  Token  9: 'half'
  Token 10: 'a'
  Token 11: 'day'
  Token 12: '.'

Total tokens: 12


In [8]:
# NLTK tokenizes text too - useful since our preprocessing function uses NLTK downstream
nltk_tokens = word_tokenize(working_sentence.text)

print("Tokens in the working sentence (NLTK):")
print()
print(nltk_tokens)

Tokens in the working sentence (NLTK):

['Battery', 'life', 'is', 'terrible', ',', 'it', 'barely', 'lasts', 'half', 'a', 'day', '.']


---
## Step 5 - Regex Cleaning

**What is regex?**

Regex stands for **Regular Expressions** - a way to describe patterns in text.

| Pattern | What it matches |
|---------|----------------|
| `\\d+` | One or more digits, such as `2024` or `42` |
| `[^a-z ]` | Anything that is NOT a lowercase letter or space |
| ` +` | One or more spaces (used to collapse extra whitespace) |

We use these patterns to strip out digits and punctuation, and to normalize whitespace,
before the text goes any further.


In [9]:
sample = working_sentence.text.strip()
print("Starting text:")
print(" ", sample)
print()

# Step A: lowercase
step_a = sample.lower()
print("After lowercasing:")
print(" ", step_a)
print()

# Step B: remove digits (\d+ matches one or more digits)
step_b = re.sub(r"\d+", "", step_a)
print("After removing digits:")
print(" ", step_b)
print()

# Step C: remove punctuation ([^a-z ] matches anything that is not a lowercase letter or space)
step_c = re.sub(r"[^a-z ]", "", step_b)
print("After removing punctuation:")
print(" ", step_c)
print()

# Step D: collapse repeated spaces
step_d = re.sub(r" +", " ", step_c).strip()
print("After collapsing whitespace:")
print(" ", step_d)

Starting text:
  Battery life is terrible, it barely lasts half a day.

After lowercasing:
  battery life is terrible, it barely lasts half a day.

After removing digits:
  battery life is terrible, it barely lasts half a day.

After removing punctuation:
  battery life is terrible it barely lasts half a day

After collapsing whitespace:
  battery life is terrible it barely lasts half a day


---
## Step 6 - Part-of-Speech (POS) Tagging

**What is it?**

POS tagging assigns a grammatical label to each word.

| Tag | Meaning | Example |
|-----|---------|---------|
| NOUN | Noun | waiter, pasta, seat |
| VERB | Verb | arrive, deliver, love |
| ADJ | Adjective | friendly, cold, bland |
| ADV | Adverb | barely, absolutely |


In [10]:
# token.pos_ gives the simple category: NOUN, VERB, ADJ...
# spacy.explain() returns a plain-English description of any tag

print("POS tags for the working sentence:")
print()
print(f"{'Token':<15} {'POS Tag':<8} {'What it means'}")
print("-" * 55)

for token in working_sentence:
    explanation = spacy.explain(token.pos_) or ""
    print(f"{token.text:<15} {token.pos_:<8} {explanation}")

POS tags for the working sentence:

Token           POS Tag  What it means
-------------------------------------------------------
Battery         NOUN     noun
life            NOUN     noun
is              AUX      auxiliary
terrible        ADJ      adjective
,               PUNCT    punctuation
it              PRON     pronoun
barely          ADV      adverb
lasts           VERB     verb
half            DET      determiner
a               DET      determiner
day             NOUN     noun
.               PUNCT    punctuation


In [11]:
# Extract all adjectives across the whole review dataset
# Adjectives carry strong sentiment signal
all_adjectives = []
for r in reviews:
    r_doc = nlp(r)
    all_adjectives.extend(token.text for token in r_doc if token.pos_ == "ADJ")

print("All adjectives across our review dataset:")
print()
print(all_adjectives)

All adjectives across our review dataset:

['excellent', 'stunning', 'low', 'terrible', 'friendly', 'cold', 'bland', 'brilliant', 'hooked', 'fast', 'neat', 'great']


In [12]:
pos_counts = Counter(
    token.pos_
    for token in working_sentence
    if not token.is_punct and not token.is_space
)

print("POS tag counts in the working sentence:")
print()
for tag, count in pos_counts.most_common():
    explanation = spacy.explain(tag) or tag
    print(f"  {tag:<8} {count:>2}  ({explanation})")

POS tag counts in the working sentence:

  NOUN      3  (noun)
  DET       2  (determiner)
  AUX       1  (auxiliary)
  ADJ       1  (adjective)
  PRON      1  (pronoun)
  ADV       1  (adverb)
  VERB      1  (verb)


---
## Step 7 - Dependency Parsing

**What is it?**

Dependency parsing draws connections between words to show which word depends on which
other word.

Every sentence has a **root** - usually the main verb. Everything else connects to the
root or to another word.


In [13]:
# token.dep_  = the type of dependency relation (nsubj, dobj, amod...)
# token.head  = the word this token connects to

print("Dependency parse for the working sentence:")
print()
print(f"{'Token':<15} {'Relation':<10} {'Head word':<15} {'Relation meaning'}")
print("-" * 70)

for token in working_sentence:
    explanation = spacy.explain(token.dep_) or ""
    print(f"{token.text:<15} {token.dep_:<10} {token.head.text:<15} {explanation}")

Dependency parse for the working sentence:

Token           Relation   Head word       Relation meaning
----------------------------------------------------------------------
Battery         compound   life            compound
life            nsubj      is              nominal subject
is              ccomp      lasts           clausal complement
terrible        acomp      is              adjectival complement
,               punct      lasts           punctuation
it              nsubj      lasts           nominal subject
barely          advmod     lasts           adverbial modifier
lasts           ROOT       lasts           root
half            predet     day             
a               det        day             determiner
day             npadvmod   lasts           noun phrase as adverbial modifier
.               punct      lasts           punctuation


/usr/local/lib/python3.13/dist-packages/spacy/glossary.py:20: UserWarning: [W118] Term 'predet' not found in glossary. It may however be explained in documentation for the corpora used to train the language. Please check `nlp.meta["sources"]` for any relevant links.
  warnings.warn(Warnings.W118.format(term=term))


In [14]:
# displacy renders a dependency arc diagram inside the notebook
from spacy import displacy

displacy.render(working_sentence.as_doc(), style="dep", jupyter=True)

---
## Step 8 - Chunking (Noun Phrase Extraction)

**What is it?**

Chunking groups consecutive words that belong together into meaningful phrases.

The most common type is **noun phrase chunking** - finding groups like `"the pasta"`
rather than just `pasta`.


In [15]:
print("Noun phrases found across the full review dataset:")
print()
print(f"{'Noun Phrase':<30} {'Root Word':<15} {'Root POS'}")
print("-" * 60)

for r in reviews:
    r_doc = nlp(r)
    for chunk in r_doc.noun_chunks:
        print(f"{chunk.text:<30} {chunk.root.text:<15} {chunk.root.pos_}")

Noun phrases found across the full review dataset:

Noun Phrase                    Root Word       Root POS
------------------------------------------------------------
The camera quality             quality         NOUN
photos                         photos          NOUN
low light                      light           NOUN
Battery life                   life            NOUN
it                             it              PRON
The waiter                     waiter          NOUN
the pasta                      pasta           NOUN
An absolutely brilliant performance performance     NOUN
me                             me              PRON
the end                        end             NOUN
Delivery                       Delivery        NOUN
packaging                      packaging       NOUN
the product                    product         NOUN


---
## Step 9 - Stopword Removal (NLTK)

Now we remove stopwords using the NLTK list we loaded in Step 2. We compare the token
count before and after, and see exactly which words got dropped.


In [16]:
nltk_stopword_set = set(nltk_stopwords)

tokens_lower = [t.lower() for t in nltk_tokens if t.isalpha()]
tokens_no_stopwords = [t for t in tokens_lower if t not in nltk_stopword_set]
removed = [t for t in tokens_lower if t in nltk_stopword_set]

print("Before stopword removal:")
print(" ", tokens_lower)
print()
print("After stopword removal:")
print(" ", tokens_no_stopwords)
print()
print("Stopwords removed:")
print(" ", removed)
print()
print(f"Token count: {len(tokens_lower)} -> {len(tokens_no_stopwords)}")

Before stopword removal:
  ['battery', 'life', 'is', 'terrible', 'it', 'barely', 'lasts', 'half', 'a', 'day']

After stopword removal:
  ['battery', 'life', 'terrible', 'barely', 'lasts', 'half', 'day']

Stopwords removed:
  ['is', 'it', 'a']

Token count: 10 -> 7


---
## Step 10 - Lemmatization (NLTK)

**What is it?**

Lemmatization reduces each word to its base dictionary form, called its **lemma**.

| Original word | Lemma |
|--------------|-------|
| arrived | arrive |
| delivers | deliver |
| performances | performance |
| brilliant | brilliant |

NLTK's `WordNetLemmatizer` needs a part-of-speech hint to do a good job - without it,
`"arrived"` would not reduce to `"arrive"`. We map spaCy's POS tags onto the simple
noun/verb/adjective/adverb categories WordNet expects.


In [17]:
from nltk.corpus import wordnet

def get_wordnet_pos(spacy_pos):
    """Maps a spaCy POS tag onto the tag format NLTK's WordNetLemmatizer expects."""
    mapping = {
        "NOUN": wordnet.NOUN,
        "VERB": wordnet.VERB,
        "AUX":  wordnet.VERB,   # auxiliary verbs like "was", "is", "has"
        "ADJ":  wordnet.ADJ,
        "ADV":  wordnet.ADV,
    }
    return mapping.get(spacy_pos, wordnet.NOUN)  # default to noun


print("Lemmatization of the working sentence (NLTK WordNetLemmatizer):")
print()
print(f"{'Original word':<15} {'Lemma':<15} {'Changed?'}")
print("-" * 45)

for token in working_sentence:
    if token.is_punct or token.is_space:
        continue
    wn_pos = get_wordnet_pos(token.pos_)
    lemma = lemmatizer.lemmatize(token.text.lower(), pos=wn_pos)
    changed = "yes" if lemma != token.text.lower() else ""
    print(f"{token.text:<15} {lemma:<15} {changed}")

Lemmatization of the working sentence (NLTK WordNetLemmatizer):

Original word   Lemma           Changed?
---------------------------------------------
Battery         battery         
life            life            
is              be              yes
terrible        terrible        
it              it              
barely          barely          
lasts           last            yes
half            half            
a               a               
day             day             


---
## Step 11 - Named Entity Recognition (Bonus)

**What is it?**

Named Entity Recognition (NER) automatically finds and classifies proper nouns - people,
organisations, locations, dates, and more. This comes for free with spaCy.


In [18]:
print("\n" + "=" * 80)
print("🔍 NAMED ENTITY RECOGNITION — CUSTOM REVIEWS")
print("=" * 80)
reviews1 = [
    "The iPhone 15 Pro is an excellent phone with a great camera and battery life.",
    "I ordered these headphones from Amazon for ₹2,499 and received them on Monday.",
    "The food at Taj Hotel in Mumbai was delicious, especially the paneer tikka.",
    "I visited Goa last December and stayed at a beautiful beach resort.",
    "The Samsung Galaxy S24 is much better than my old phone.",
]
entity_count = 0
for i, review in enumerate(reviews1, start=1):
    print(f"\n📝 Review {i}")
    print("-" * 80)
    print(review)
    doc = nlp(review)
    print("\n🏷️ Named Entities:")
    if doc.ents:
        print(f"{'Entity':<25} {'Label':<12} {'Description'}")
        print("-" * 70)
        for ent in doc.ents:
            entity_count += 1
            explanation = spacy.explain(ent.label_) or "No description available"
            print(
                f"{ent.text:<25} "
                f"{ent.label_:<12} "
                f"{explanation}"
            )
    else:
        print("   No named entities found.")
print("\n" + "=" * 80)
print(f"📊 Total named entities found: {entity_count}")
print("=" * 80)


🔍 NAMED ENTITY RECOGNITION — CUSTOM REVIEWS

📝 Review 1
--------------------------------------------------------------------------------
The iPhone 15 Pro is an excellent phone with a great camera and battery life.

🏷️ Named Entities:
Entity                    Label        Description
----------------------------------------------------------------------
15                        CARDINAL     Numerals that do not fall under another type

📝 Review 2
--------------------------------------------------------------------------------
I ordered these headphones from Amazon for ₹2,499 and received them on Monday.

🏷️ Named Entities:
Entity                    Label        Description
----------------------------------------------------------------------
Amazon                    ORG          Companies, agencies, institutions, etc.
2,499                     MONEY        Monetary values, including unit
Monday                    DATE         Absolute or relative dates or periods

📝 Review 3
-----

---
## Step 12 - The Full Pipeline: One Function

Now we combine cleaning, tokenization, stopword removal, and lemmatization into a single
reusable function. This is what a real preprocessing step looks like before text is fed
into a downstream model such as a bag-of-words vectorizer or an embedding model.


In [19]:
def preprocess_text(raw_text):
    """Runs the complete NLTK-based text preprocessing pipeline on raw text.

    Steps:
      1. Remove HTML tags and URLs
      2. Lowercase everything
      3. Remove digits and punctuation
      4. Tokenize (NLTK)
      5. Remove stopwords (NLTK)
      6. Lemmatize each remaining token (NLTK WordNetLemmatizer, POS-aware)

    Returns a dict with the processed text plus the intermediate token lists,
    so we can inspect every stage.
    """
    # Step 1: strip HTML tags and URLs
    text = re.sub(r"<.*?>", " ", raw_text)
    text = re.sub(r"http\S+", "", text)

    # Step 2-3: lowercase, remove digits, remove punctuation, collapse whitespace
    text = text.lower()
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^a-z ]", "", text)
    text = re.sub(r" +", " ", text).strip()

    # Step 4: tokenize
    tokens = word_tokenize(text)

    # Step 5: remove stopwords
    tokens_no_stop = [t for t in tokens if t not in nltk_stopword_set]

    # Step 6: POS-tag (via spaCy, for accurate lemmatization) then lemmatize with NLTK
    pos_doc = nlp(" ".join(tokens_no_stop))
    lemmas = [
        lemmatizer.lemmatize(t.text, pos=get_wordnet_pos(t.pos_))
        for t in pos_doc
    ]

    processed_text = " ".join(lemmas)

    return {
        "original": raw_text,
        "tokens": tokens,
        "tokens_no_stopwords": tokens_no_stop,
        "lemmas": lemmas,
        "processed_text": processed_text,
    }


# Run the full pipeline on every review in our dataset
print("Running the full preprocessing pipeline on all reviews:")
print("=" * 70)

for i, review in enumerate(reviews, start=1):
    result = preprocess_text(review)
    print(f"\nReview {i}")
    print(f"  Original : {result['original']}")
    print(f"  Processed: {result['processed_text']}")

Running the full preprocessing pipeline on all reviews:

Review 1
  Original : The camera quality is excellent and photos look stunning in low light.
  Processed: camera quality excellent photo look stunning low light

Review 2
  Original : Battery life is terrible, it barely lasts half a day.
  Processed: battery life terrible barely last half day

Review 3
  Original : The waiter was friendly but the pasta arrived cold and bland.
  Processed: waiter friendly pasta arrive cold bland

Review 4
  Original : An absolutely brilliant performance kept me hooked until the end.
  Processed: absolutely brilliant performance keep hooked end

Review 5
  Original : Delivery was fast, packaging was neat, and the product works great.
  Processed: delivery fast packaging neat product work great


---
## Summary - What We Covered in Demo 1

We took our custom review dataset and ran it through a complete NLP preprocessing
pipeline.

| Step | What we did | Tool |
|------|------------|------|
| 0 | Loaded spaCy's English model and NLTK data | spaCy, NLTK |
| 1 | Defined our own review dataset | - |
| 2 | Loaded and inspected the NLTK stopword list | NLTK |
| 3 | Split text into sentences | spaCy |
| 4 | Tokenized text into words | spaCy, NLTK |
| 5 | Cleaned text with regex (lowercase, digits, punctuation) | `re` |
| 6 | Tagged each word's part of speech | spaCy |
| 7 | Parsed grammatical dependencies | spaCy |
| 8 | Extracted noun phrases (chunking) | spaCy |
| 9 | Removed stopwords | NLTK |
| 10 | Lemmatized words to their base form | NLTK (WordNet) |
| 11 | Recognized named entities | spaCy |
| 12 | Combined everything into one `preprocess_text()` function | NLTK + `re` |

This is exactly the kind of pipeline that sits in front of the Bag-of-Words, TF-IDF, and
embedding techniques covered in the rest of the session - and it's the same review
dataset we'll carry into the Transformer applications demo.
